In [7]:
# suppress Python warnings
import warnings
warnings.filterwarnings("ignore")

# suppress all library loggers by default
import logging
logging.getLogger().setLevel(logging.ERROR)

# in particular, silence lightgbm’s C++ logger
logging.getLogger("lightgbm").setLevel(logging.ERROR)

# silence optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)

In [9]:
# ─── 1) Imports, config & data ───────────────────────────────────────────────────

import time
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

import lightgbm as lgb
import optuna
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import gp_minimize
from skopt.space import Real, Integer

# unified config
cfg = {
    "general": {
        "seeds":    [0, 42, 7, 123, 999],
        "n_trials": 30,            # cut down for speed if needed
    },
    "lgbm": {
        "n_estimators":           1000,
        "early_stopping_rounds":  50,
        "search_space": {
            "learning_rate":   {"low": 1e-3, "high": 1e-1},
            "max_depth":       {"low": 3,    "high": 12},
            "subsample":       {"low": 0.5,  "high": 1.0},
            "colsample_bytree":{"low": 0.5,  "high": 1.0},
        },
    },
}

# load your splits
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te = joblib.load("data/processed/test.pkl")


# ─── 2) Common callbacks to silence logs & early-stop ─────────────────────────────

callbacks = [
    # silent early stopping
    lgb.early_stopping(stopping_rounds=cfg["lgbm"]["early_stopping_rounds"]),
    # no per-iter logging
    lgb.log_evaluation(period=0),
]


# ─── 3) Tuning functions ────────────────────────────────────────────────────────

def tune_optuna_lgb(seed):
    def obj(trial):
        p = {
            "objective":        "regression",
            "metric":           "rmse",
            "random_state":     seed,
            "learning_rate":    trial.suggest_float(
                                     "learning_rate",
                                     **cfg["lgbm"]["search_space"]["learning_rate"],
                                     log=True
                                 ),
            "max_depth":        trial.suggest_int(
                                     "max_depth",
                                     **cfg["lgbm"]["search_space"]["max_depth"]
                                 ),
            "subsample":        trial.suggest_float(
                                     "subsample",
                                     **cfg["lgbm"]["search_space"]["subsample"]
                                 ),
            "colsample_bytree": trial.suggest_float(
                                     "colsample_bytree",
                                     **cfg["lgbm"]["search_space"]["colsample_bytree"]
                                 ),
        }
        dtr  = lgb.Dataset(X_tr,  label=y_tr)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtr)
        bst = lgb.train(
            p,
            dtr,
            num_boost_round=cfg["lgbm"]["n_estimators"],
            valid_sets=[dval],
            callbacks=callbacks
        )
        pred = bst.predict(X_val, num_iteration=bst.best_iteration)
        return mean_squared_error(y_val, pred, squared=False)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(obj, n_trials=cfg["general"]["n_trials"], show_progress_bar=False)
    return study.best_params, study.best_value


def tune_hyperopt_lgb(seed):
    space = {
        "learning_rate":   hp.loguniform(
                               "learning_rate",
                               np.log(cfg["lgbm"]["search_space"]["learning_rate"]["low"]),
                               np.log(cfg["lgbm"]["search_space"]["learning_rate"]["high"])
                           ),
        "max_depth":       hp.quniform(
                               "max_depth",
                               cfg["lgbm"]["search_space"]["max_depth"]["low"],
                               cfg["lgbm"]["search_space"]["max_depth"]["high"],
                               1
                           ),
        "subsample":       hp.uniform(
                               "subsample",
                               cfg["lgbm"]["search_space"]["subsample"]["low"],
                               cfg["lgbm"]["search_space"]["subsample"]["high"]
                           ),
        "colsample_bytree":hp.uniform(
                               "colsample_bytree",
                               cfg["lgbm"]["search_space"]["colsample_bytree"]["low"],
                               cfg["lgbm"]["search_space"]["colsample_bytree"]["high"]
                           ),
    }
    trials = Trials()
    def obj(params):
        p = {
            "objective":        "regression",
            "metric":           "rmse",
            "random_state":     seed,
            **{k: int(params[k]) if k=="max_depth" else params[k]
               for k in params}
        }
        dtr  = lgb.Dataset(X_tr,  label=y_tr)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtr)
        bst = lgb.train(
            p,
            dtr,
            num_boost_round=cfg["lgbm"]["n_estimators"],
            valid_sets=[dval],
            callbacks=callbacks
        )
        pred = bst.predict(X_val, num_iteration=bst.best_iteration)
        rmse = mean_squared_error(y_val, pred, squared=False)
        return {"loss": rmse, "status": STATUS_OK}

    best = fmin(
        fn=obj,
        space=space,
        algo=tpe.suggest,
        max_evals=cfg["general"]["n_trials"],
        trials=trials,
        rstate=np.random.default_rng(seed)
    )
    return (
        {
          "learning_rate":    best["learning_rate"],
          "max_depth":        int(best["max_depth"]),
          "subsample":        best["subsample"],
          "colsample_bytree": best["colsample_bytree"],
          "random_state":     seed
        },
        np.min(trials.losses())
    )


def tune_skopt_lgb(seed):
    dims = [
        Real(**cfg["lgbm"]["search_space"]["learning_rate"], prior="log-uniform", name="learning_rate"),
        Integer(**cfg["lgbm"]["search_space"]["max_depth"],           name="max_depth"),
        Real(**cfg["lgbm"]["search_space"]["subsample"],             name="subsample"),
        Real(**cfg["lgbm"]["search_space"]["colsample_bytree"],      name="colsample_bytree"),
    ]
    def obj(x):
        p = {
            "objective":        "regression",
            "metric":           "rmse",
            "random_state":     seed,
            "learning_rate":    x[0],
            "max_depth":        x[1],
            "subsample":        x[2],
            "colsample_bytree": x[3],
        }
        dtr  = lgb.Dataset(X_tr,  label=y_tr)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtr)
        bst = lgb.train(
            p,
            dtr,
            num_boost_round=cfg["lgbm"]["n_estimators"],
            valid_sets=[dval],
            callbacks=callbacks
        )
        pred = bst.predict(X_val, num_iteration=bst.best_iteration)
        return mean_squared_error(y_val, pred, squared=False)

    res = gp_minimize(
        func=obj,
        dimensions=dims,
        n_calls=cfg["general"]["n_trials"],
        random_state=seed,
        verbose=False
    )
    return (
        {
          "learning_rate":    res.x[0],
          "max_depth":        res.x[1],
          "subsample":        res.x[2],
          "colsample_bytree": res.x[3],
          "random_state":     seed
        },
        res.fun
    )


# ─── 4) Run all seeds × methods & save tuning table ─────────────────────────────

methods = [("Optuna", tune_optuna_lgb),
           ("Hyperopt", tune_hyperopt_lgb),
           ("Skopt", tune_skopt_lgb)]

rows = []
for pkg, fn in methods:
    for sd in cfg["general"]["seeds"]:
        best_p, rmse = fn(sd)
        rows.append(dict(package=pkg, seed=sd, val_rmse=rmse, **best_p))

df_tune = pd.DataFrame(rows)
df_tune.to_csv("rq1_lgbm_tuning.csv", index=False)
print("→ tuning results → rq1_lgbm_tuning.csv")


# ─── 5) Re‐fit on train+val, test & save final metrics ──────────────────────────

best_row = df_tune.loc[df_tune.val_rmse.idxmin()]
final_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "n_estimators":     cfg["lgbm"]["n_estimators"],
    "learning_rate":    best_row.learning_rate,
    "max_depth":        int(best_row.max_depth),
    "subsample":        best_row.subsample,
    "colsample_bytree": best_row.colsample_bytree,
    "random_state":     int(best_row.random_state),
}

# fit on full train+val
X_full = np.vstack([X_tr, X_val])
y_full = np.concatenate([y_tr, y_val])
model = lgb.LGBMRegressor(**final_params)

t0 = time.time()
model.fit(
    X_full, y_full,
    callbacks=callbacks
)
train_time = time.time() - t0

# test
t1 = time.time()
pred = model.predict(X_te)
pred_time = time.time() - t1

df_final = pd.DataFrame([{
    **final_params,
    "train_time_s": round(train_time,3),
    "pred_time_s":  round(pred_time,3),
    "test_rmse":    round(mean_squared_error(y_te, pred, squared=False),3),
    "test_r2":      round(r2_score(y_te, pred),4),
}])
df_final.to_csv("rq1_lgbm_final.csv", index=False)
print("→ final results → rq1_lgbm_final.csv")

[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=10) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=1024) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=10) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=1024) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010561 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1656
[LightGBM] [Info] Number of data points in the train set: 151075, number of used features: 129
[LightGBM] [Warning] Provided paramet

ValueError: cannot convert float NaN to integer

In [21]:
import pandas as pd
import numpy as np
import time
import joblib
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation

# ── 1) Reload tuning results & data ─────────────────────────────────────────────
df_tune = pd.read_csv("rq1_lgbm_tuning.csv")
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te = joblib.load("data/processed/test.pkl")

# ── 2) Pick best row ───────────────────────────────────────────────────────────
best = df_tune.loc[df_tune.val_rmse.idxmin()]
seed = int(best.seed)

# ── 3) Build final parameters ─────────────────────────────────────────────────
final_params = {
    "objective":         "regression",
    "metric":            "rmse",
    "n_estimators":      cfg["lgbm"]["n_estimators"],
    "learning_rate":     best.learning_rate,
    "max_depth":         int(best.max_depth),
    "subsample":         best.subsample,
    "colsample_bytree":  best.colsample_bytree,
    "random_state":      seed,
    "verbosity":         -1
}

# ── 4) Retrain on train+val ────────────────────────────────────────────────────
X_full = np.vstack([X_tr, X_val])
y_full = np.concatenate([y_tr, y_val])

model = lgb.LGBMRegressor(**final_params)

t0 = time.time()
model.fit(
    X_full, y_full,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[
        early_stopping(stopping_rounds=cfg["lgbm"]["early_stopping_rounds"]),
        log_evaluation(period=0)
    ]
)
train_time = time.time() - t0

# ── 5) Final test & summary ───────────────────────────────────────────────────
t1 = time.time()
y_pred = model.predict(X_te)
pred_time = time.time() - t1

test_rmse = mean_squared_error(y_te, y_pred, squared=False)
test_r2   = r2_score(y_te, y_pred)

df_final = pd.DataFrame([{
    **final_params,
    "train_time_s": round(train_time, 3),
    "pred_time_s":  round(pred_time, 3),
    "test_rmse":    round(test_rmse, 3),
    "test_r2":      round(test_r2, 4)
}])

# ── 6) Save ────────────────────────────────────────────────────────────────────
df_final.to_csv("rq1_lgbm_final.csv", index=False)
print("→ final results saved to rq1_lgbm_final.csv")

df_final

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 3061.48
→ final results saved to rq1_lgbm_final.csv


,objective,metric,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,random_state,verbosity,train_time_s,pred_time_s,test_rmse,test_r2
0,regression,rmse,1000,0.088844,12,0.660001,0.996979,7,-1,3.211,0.393,6878.613,0.9158
